In [11]:
import torch 
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [12]:
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

trainset = CIFAR10(root="./data", train=True, download=True, transform=transform)
testset = CIFAR10(root="./data", train=False, download=True, transform=transform)

trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader = DataLoader(trainset, batch_size=64)

Files already downloaded and verified
Files already downloaded and verified


Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [15]:
type(trainset)


torchvision.datasets.cifar.CIFAR10

In [4]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        
        self.con_layers = nn.Sequential(
                    nn.Conv2d(3,32,  kernel_size=3, padding=1),
                    nn.ReLU(),
                    nn.MaxPool2d(2,2),
            
                    nn.Conv2d(32,64, kernel_size=3, padding=1),
                    nn.ReLU(),
                    nn.MaxPool2d(2,2),

                    nn.Conv2d(64, 128, kernel_size=3, padding=1),
                    nn.ReLU(),
                    nn.MaxPool2d(2,2)
        )


        self.fc_layers=nn.Sequential(
                    nn.Linear(4*4*128, 256),
                    nn.ReLU(),

                    nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.con_layers(x)
        
        x = x.view(x.size(0), -1)
        
        x = self.fc_layers(x)

        return x

In [5]:
model = CNN()

In [6]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

# Training the CNN

In [8]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()
        
        outputs = model.forward(images)#FP
        loss=criterion(outputs, labels)#loss fnx
        loss.backward()#BP
        optimizer.step()#UPDATE PARAMS
        epoch_training_loss+=loss.item()
    print(f"epoch={epoch} & loss={epoch_training_loss/len(trainloader)}")

epoch=0 & loss=1.4015134556214217
epoch=1 & loss=0.9744862802040851
epoch=2 & loss=0.7933293936959923
epoch=3 & loss=0.6668728045033067
epoch=4 & loss=0.5627075089403736
epoch=5 & loss=0.4723362652846919
epoch=6 & loss=0.3900443233378098
epoch=7 & loss=0.3144019304791375
epoch=8 & loss=0.24979805858696208
epoch=9 & loss=0.2012162356758895


In [17]:
correct_labels = 0
total_labels = 0
model.eval()

with torch.no_grad():
    for images, labels in testloader:
        outputs=model.forward(images)
        _,predicted = torch.max(outputs, 1)


        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"accuracy = {correct_labels / total_labels*100} * 100")

accuracy = 95.04 * 100
